# MergeSort e Merge (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/MergeSort_cpp.ipynb)

Capitolo 17 — Algoritmi ricorsivi. Notebook eseguibile di `MergeSort` e `Merge`: gli algoritmi in C++, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/MergeSort_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/MergeSort_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `RecursiveSum` (PascalCase).

## Merge Sort e Merge

- **MergeSort**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: il sotto-array $a[l..r]$ ordinato in modo non decrescente. Si chiama `MergeSort(a, 1, n)` nello pseudocodice, `merge_sort(a, 0, n-1)` nel codice.
- **Merge**$(a,l,m,r)$ — *Input*: un array e indici $l\le m<r$ tali che $a[l..m]$ e $a[m+1..r]$ siano ordinati. *Output*: $a[l..r]$ ordinato. Copia le due metà in $L$ e $R$ e riversa in $a$, passo dopo passo, il minore fra i primi due elementi non ancora copiati.

Il contatore `comparisons` conta i confronti fra elementi eseguiti da `Merge`. Sull'array dell'esempio $a=[9,12,3,16,19,11,12]$ si stampano le chiamate di `Merge` nell'ordine di esecuzione (con l'array dopo ciascuna, tabella del capitolo; gli indici sono da 0: aggiungendo 1 si ritrovano quelli del capitolo), poi la traccia di `Merge(a, 0, 1, 3)` su $a=[5,8,2,7]$ (esempio dell'implementazione in C++).

In [1]:
%%writefile merge_sort.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // confronti fra elementi eseguiti da Merge
int msCalls = 0;             // chiamate di MergeSort

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r): fonde a[l..m] e a[m+1..r], entrambi ordinati
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;             // dimensione della parte sinistra
    int n2 = r - m;                 // dimensione della parte destra
    vector<int> L(n1), R(n2);       // array ausiliari
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)        // fonde L e R in a[l..r]
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)                  // copia il resto di L
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)                  // copia il resto di R
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

// MergeSort(a, l, r): ordina a[l..r]
void MergeSort(int *a, int l, int r)
{
    msCalls++;
    if (l < r)
    {
        int m = l + (r - l) / 2;    // divide
        MergeSort(a, l, m);         // conquer (metà sinistra)
        MergeSort(a, m + 1, r);     // conquer (metà destra)
        Merge(a, l, m, r);          // combine
    }
}

// ---- varianti che stampano la traccia ----
void MergeSortLog(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSortLog(a, l, m);
        MergeSortLog(a, m + 1, r);
        string left = Fmt(a + l, m - l + 1), right = Fmt(a + m + 1, r - m);
        Merge(a, l, m, r);
        printf("Merge(a,%d,%d,%d)  %s e %s  ->  %s\n", l, m, r, left.c_str(), right.c_str(), Fmt(a, 7).c_str());
    }
}

void MergeTrace(int *a, int n, int l, int m, int r)
{
    int n1 = m - l + 1, n2 = r - m;
    vector<int> L(a + l, a + m + 1), R(a + m + 1, a + r + 1);
    printf("Merge(a, %d, %d, %d) su a = %s\n", l, m, r, Fmt(a, n).c_str());
    printf("n1=%d, n2=%d, L=%s, R=%s\n", n1, n2, Fmt(L.data(), n1).c_str(), Fmt(R.data(), n2).c_str());
    int i = 0, j = 0, k = l, it = 0;
    while (i < n1 && j < n2)
    {
        it++;
        char head[200];
        if (L[i] <= R[j])
        {
            snprintf(head, sizeof head, "%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? Sì => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
            a[k] = L[i];
            i++;
        }
        else
        {
            snprintf(head, sizeof head, "%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
            a[k] = R[j];
            j++;
        }
        k++;
        printf("%s, i=%d, j=%d, k=%d; a=%s\n", head, i, j, k, Fmt(a, n).c_str());
    }
    while (i < n1)
    {
        a[k] = L[i];
        printf("    ciclo 2 (resto di L): a[%d] = L[%d] = %d; a=%s\n", k, i, L[i], Fmt(a, n).c_str());
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        printf("    ciclo 3 (resto di R): a[%d] = R[%d] = %d; a=%s\n", k, j, R[j], Fmt(a, n).c_str());
        j++;
        k++;
    }
    printf("risultato a = %s, confronti = %d\n", Fmt(a, n).c_str(), it);
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 12, 3, 16, 19, 11, 12};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("Chiamate di Merge in ordine di esecuzione:\n");
    int copia[] = {9, 12, 3, 16, 19, 11, 12};
    MergeSortLog(copia, 0, n - 1);
    comparisons = 0;
    msCalls = 0;
    MergeSort(a, 0, n - 1);
    printf("ordinato = %s   chiamate di MergeSort=%d (2n-1=%d)   confronti=%lld\n", Fmt(a, n).c_str(), msCalls, 2 * n - 1, comparisons);
    printf("\n");
    int b[] = {5, 8, 2, 7};
    MergeTrace(b, 4, 0, 1, 3);
    return 0;
}
#endif

Writing merge_sort.cpp


In [2]:
!g++ -std=c++17 -O2 -o merge_sort merge_sort.cpp && ./merge_sort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Chiamate di Merge in ordine di esecuzione:
Merge(a,0,0,1)  [9] e [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] e [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] e [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] e [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] e [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] e [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
ordinato = [3, 9, 11, 12, 12, 16, 19]   chiamate di MergeSort=13 (2n-1=13)   confronti=14

Merge(a, 0, 1, 3) su a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. ciclo 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. ciclo 1: L[0]=5 <= R[1]=7? Sì => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. ciclo 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    ciclo 2 (resto di L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
risultato a = [2, 5, 7, 8], confronti = 3


**Prova tu.** Ordina un altro array: il numero di confronti di `Merge` è al massimo $n\lceil\log_2 n\rceil$ e le chiamate di `MergeSort` sono sempre $2n-1$.

In [3]:
%%writefile prova_merge_sort.cpp
#define NOMAIN
#include "merge_sort.cpp"

int main()
{
    // Ordina un altro array e conta i confronti (al massimo n*ceil(log2 n)).
    int b[] = {38, 27, 43, 3, 9, 82, 10, 5};
    int n = 8;
    comparisons = 0;
    msCalls = 0;
    MergeSort(b, 0, n - 1);
    printf("%s %lld %d\n", Fmt(b, n).c_str(), comparisons, msCalls);
    return 0;
}

Writing prova_merge_sort.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_merge_sort prova_merge_sort.cpp && ./prova_merge_sort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).